# Step 1 — Ingest Raw Data to Bronze

**Purpose:** Load raw CSV files from the Unity Catalog Volume into Delta tables in the
`bronze` schema. This is the first stage of the Medallion Architecture.

### What This Notebook Does
1. Reads each CSV file from the `raw_landing_zone` volume **without** schema inference
   (all columns stored as strings to preserve raw fidelity).
2. Adds an `ingest_timestamp` metadata column for lineage tracking.
3. Writes each dataset as an **overwrite** Delta table into `medallion_trial.bronze`.

### Source Datasets (Olist Brazilian E-Commerce)
| CSV File | Bronze Table |
|----------|-------------|
| `olist_orders_dataset.csv` | `raw_orders` |
| `olist_order_items_dataset.csv` | `raw_order_items` |
| `olist_customers_dataset.csv` | `raw_customers` |
| `olist_products_dataset.csv` | `raw_products` |

> **Prerequisites:** Run `schema_mgt/catalog_and_schema_creation` first and upload
> the CSV files to the `raw_landing_zone` volume.

---

In [0]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
# current_timestamp() is used to stamp each row with the time of ingestion,
# which is a Bronze-layer best practice for auditing and lineage.
from pyspark.sql.functions import current_timestamp

In [0]:
# ---------------------------------------------------------------------------
# Configuration — edit these if you use a different catalog or schema name
# ---------------------------------------------------------------------------
catalog     = "medallion_trial"                            # Unity Catalog name
schema      = "bronze"                                     # Target schema
volume_path = f"/Volumes/{catalog}/{schema}/raw_landing_zone"  # Path to raw CSVs

In [0]:
# ---------------------------------------------------------------------------
# Dataset mapping: CSV filename -> Bronze Delta table name
# ---------------------------------------------------------------------------
# Each key is the name of a CSV file in the raw_landing_zone volume.
# Each value is the name of the Delta table that will be created in Bronze.
datasets = {
    "olist_orders_dataset.csv":      "raw_orders",
    "olist_order_items_dataset.csv": "raw_order_items",
    "olist_customers_dataset.csv":   "raw_customers",
    "olist_products_dataset.csv":    "raw_products",
}

In [0]:
def load_csv_to_bronze(file_name: str, table_name: str) -> None:
    """
    Read a single CSV file from the landing zone and write it as a
    Delta table in the Bronze schema.

    Parameters
    ----------
    file_name  : str  — Name of the CSV file inside the volume.
    table_name : str  — Target Delta table name in the Bronze schema.

    Key design decisions
    --------------------
    - inferSchema = False  -> keeps every column as a string so the Bronze
      layer is a faithful copy of the source; type casting happens in Silver.
    - ingest_timestamp     -> added for data lineage and freshness tracking.
    - mode = overwrite     -> full-refresh pattern; swap to 'append' for
      incremental ingestion in production.
    """
    print(f"Ingesting {file_name} -> {catalog}.{schema}.{table_name} ...")

    # 1. Read the raw CSV — all columns as strings (no schema inference)
    df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .load(f"{volume_path}/{file_name}")
    )

    # 2. Add an ingestion timestamp for auditing and lineage
    df_with_metadata = df.withColumn("ingest_timestamp", current_timestamp())

    # 3. Persist as a managed Delta table in Unity Catalog
    (
        df_with_metadata.write.format("delta")
        .mode("overwrite")
        .saveAsTable(f"{catalog}.{schema}.{table_name}")
    )

    print(f"SUCCESS: {table_name} created.\n")

In [0]:
# ---------------------------------------------------------------------------
# Run the ingestion for every dataset in the mapping
# ---------------------------------------------------------------------------
for file, table in datasets.items():
    load_csv_to_bronze(file, table)

print("Bronze layer ingestion complete.")

---
### Quick Validation
Preview the `raw_orders` table to confirm data landed correctly.

In [0]:
%sql
-- Quick sanity check: preview the first 5 rows of the ingested orders table
SELECT * FROM medallion_trial.bronze.raw_orders LIMIT 5